# dr_evt 02: the Python streaming API, one platform

notebook 01 ran the batch CLI. This session drives one cluster from Python, one job at a
time, the way our auction client will drive every platform. By the end you will know the
four calls that matter, the one idiom that trips everyone, what the monitoring queries
return, and exactly what the in-process API cannot give us (which is why a per-job timing accessor is the first extension this branch adds).

The bindings are `python/dr_evt_bindings.cpp`. `python/README.md` is stale (it documents
`insert_job`, `run_until_inclusive`, `get_wait_queue_size`, none of which exist); trust
the bindings file and `tests/test_python_api.py`.

In [1]:
from pathlib import Path
import sys, os, tempfile
import pandas as pd

DR_EVT = Path.cwd().resolve()                        # run from learn/ or from the repository root
while not (DR_EVT / "CMakeLists.txt").exists() and DR_EVT != DR_EVT.parent:
    DR_EVT = DR_EVT.parent
assert (DR_EVT / "CMakeLists.txt").exists(), "run this notebook from learn/ inside a dr_evt checkout"
INSTALL = Path(os.environ.get("DR_EVT_INSTALL", DR_EVT / "install"))   # the cmake install prefix
OUT = Path.cwd() / "output"                                            # scratch space, gitignored
OUT.mkdir(exist_ok=True)
sys.path.insert(0, str(INSTALL / "lib" / "python"))   # the module is not in site-packages

import dr_evt
QUEUE = QUEUE if getattr(dr_evt, "legacy_queue_input", True) else "1"   # numeric queue ids by default
print("dr_evt", dr_evt.__version__, "| legacy queue names:", getattr(dr_evt, "legacy_queue_input", "n/a"), "| queue value:", QUEUE)
print("module:    ", [n for n in dir(dr_evt) if not n.startswith("_")])
print("Simulation:", [n for n in dir(dr_evt.Simulation) if not n.startswith("_")])

dr_evt 1.0.0 | legacy queue names: False | queue value: 1
module:     ['ACTUAL', 'BackfillPolicy', 'BackfillWindow', 'CONSERVATIVE', 'DISTRIBUTION', 'EASY', 'FCFS', 'FCFS_CONSERVATIVE', 'JobAppendRequest', 'LIMIT', 'LJF', 'NONE', 'PriorityPolicy', 'ResourceRelease', 'RunTimeMode', 'SJF', 'SimParams', 'Simulation', 'Statistics', 'legacy_queue_input']
Simulation: ['advance_to', 'append_job', 'append_jobs', 'get_active_job_count', 'get_available_nodes', 'get_backfill_window', 'get_current_time', 'get_fcfs_head_shadow_time', 'get_nodes_in_use', 'get_statistics', 'get_trace_size', 'initialize_trace', 'print_stats', 'run', 'run_until_exclusive', 'write_simulated_trace']


## SimParams: the knobs you can set from Python

Eight fields are bound. Notably **not** bound: `seed`, `outfile`, `resource_trace`,
`msec_output`, `max_time`, the distribution parameters. Two consequences we will see below:
`write_simulated_trace()` is a silent no-op from Python (there is no output file to write
to), and the only randomness (`distribution` runtime mode) cannot be seeded from Python.

The default `total_nodes` is 795: dr_evt was written for Lassen.

In [2]:
p = dr_evt.SimParams()
{k: getattr(p, k) for k in ["infile", "total_nodes", "trace_format", "timestamp_format",
                             "run_time_mode", "backfill_policy", "priority_policy", "verbose"]}

{'infile': '',
 'total_nodes': 795,
 'trace_format': 'simple',
 'timestamp_format': 'iso',
 'run_time_mode': <RunTimeMode.ACTUAL: 0>,
 'backfill_policy': <BackfillPolicy.EASY: 0>,
 'priority_policy': <PriorityPolicy.FCFS: 0>,
 'verbose': False}

One more gotcha before the first `Simulation`: constructing one **parses the header of
`infile`** to learn the column layout, even in streaming mode where no rows are ever read.
With `infile` empty the constructor raises `Failed to initialize data columns`. Every
example in the repo quietly sidesteps this by pointing `infile` at a real trace. We point it
at a header-only file instead, which makes the dependency explicit.

In [3]:
HEADER_ONLY = OUT / "header_only.csv"
HEADER_ONLY.write_text("job_submit_time,num_nodes,queue,time_limit\n")

def make_params(total_nodes, backfill=dr_evt.BackfillPolicy.EASY, priority=dr_evt.PriorityPolicy.FCFS):
    p = dr_evt.SimParams()
    p.infile = str(HEADER_ONLY)        # header is parsed at construction; rows never are
    p.total_nodes = total_nodes
    p.trace_format = "simple"
    p.timestamp_format = "epoch"
    p.run_time_mode = dr_evt.RunTimeMode.LIMIT   # appended jobs run exactly their limit regardless (see below)
    p.backfill_policy = backfill
    p.priority_policy = priority
    return p

params = make_params(100)
sim = dr_evt.Simulation(params)   # keep `params` alive: the C++ Simulation holds a reference to it
print("t =", sim.get_current_time(), "| free =", sim.get_available_nodes(), "| waiting =", sim.get_active_job_count())

t = 0.0 | free = 100 | waiting = 0


## The four calls and the one idiom

- `append_job(submit_time, num_nodes, queue, limit_time) -> job_idx` creates the record
  and puts it in the wait queue. `queue` is a digit string (`"1"`) unless the build reads
  legacy queue names. `submit_time` must be `>= current_time`. The returned
  `job_idx` is a permanent, monotone id.
- `append_jobs([JobAppendRequest, ...]) -> [job_idx]` is the batch form; submit times
  must be non-decreasing; all-or-nothing on validation.
- `advance_to(T)` processes every event with time `<= T`, makes scheduling decisions, and
  sets `current_time = T`. That last part is unconditional: a platform clock only moves
  forward.
- The monitoring getters read state at `current_time`.

The idiom: **appending only queues; the scheduler runs on the next `advance_to`**. A job
appended at the current time does not start until you call `advance_to(current_time)`
again. Watch the counters.

In [4]:
idx = sim.append_job(0.0, 20, QUEUE, 200.0)
print(f"appended job_idx={idx} at t=0 -> in use = {sim.get_nodes_in_use()}, waiting = {sim.get_active_job_count()}")
sim.advance_to(0.0)
print(f"advance_to(0)              -> in use = {sim.get_nodes_in_use()}, waiting = {sim.get_active_job_count()}")
sim.append_job(10.0, 30, QUEUE, 150.0)     # a future arrival is fine: 10 >= current_time 0
print(f"appended a t=10 job        -> waiting = {sim.get_active_job_count()} (not eligible yet: t is still {sim.get_current_time()})")
sim.advance_to(10.0)
print(f"advance_to(10)             -> in use = {sim.get_nodes_in_use()}, t = {sim.get_current_time()}")

appended job_idx=0 at t=0 -> in use = 0, waiting = 1
advance_to(0)              -> in use = 20, waiting = 0
appended a t=10 job        -> waiting = 0 (not eligible yet: t is still 0.0)
advance_to(10)             -> in use = 50, t = 10.0


## Stepping through notebook 01's trace with the monitoring API

Same ten jobs as the CLI run. After each arrival we advance to it and read the
**backfill window**: `available_nodes`, the FCFS head's `shadow_time` (`-1` when nothing
waits), and the projected `releases` (time, nodes) between now and the shadow time,
computed from running jobs' time limits. This snapshot is the read-only oracle our auction
will use to predict where a job would start.

In [5]:
trace = pd.read_csv(DR_EVT / "python" / "examples" / "sample_trace.csv")
params = make_params(100)
sim = dr_evt.Simulation(params)
log = []
for i, row in trace.iterrows():
    t = float(row.job_submit_time)
    sim.append_job(t, int(row.num_nodes), QUEUE, float(row.time_limit))
    sim.advance_to(t)
    w = sim.get_backfill_window()
    log.append(dict(t=t, job=i, nodes=int(row.num_nodes), limit=int(row.time_limit),
                    in_use=sim.get_nodes_in_use(), free=w.available_nodes,
                    waiting=sim.get_active_job_count(), shadow=w.shadow_time,
                    releases=[(r.time, r.nodes_released) for r in w.releases]))
pd.DataFrame(log)

,t,job,nodes,limit,in_use,free,waiting,shadow,releases
0,0.0,0,20,200,20,80,0,-1.0,[]
1,10.0,1,30,150,50,50,0,-1.0,[]
2,20.0,2,15,300,65,35,0,-1.0,[]
3,30.0,3,40,100,65,35,1,160.0,"[(160.0, 30)]"
4,40.0,4,25,250,65,35,2,160.0,"[(160.0, 30)]"
5,50.0,5,10,80,75,25,2,160.0,"[(130.0, 10), (160.0, 30)]"
6,60.0,6,35,180,75,25,3,160.0,"[(130.0, 10), (160.0, 30)]"
7,70.0,7,60,220,75,25,4,160.0,"[(130.0, 10), (160.0, 30)]"
8,80.0,8,20,90,75,25,5,160.0,"[(130.0, 10), (160.0, 30)]"
9,90.0,9,45,160,75,25,6,160.0,"[(130.0, 10), (160.0, 30)]"


Read the rows against the hand walkthrough from notebook 01:

- t=30: job 3 (40 nodes) cannot fit in 35; `shadow` becomes 160 and `releases` lists the
  one projected release that gets there: job 1's 30 nodes at 160.
- t=40: job 4 fits (35 free) but 40 + 250 is not < 160, so it waits; `waiting` is 2.
- t=50: job 5 backfills (50 + 80 < 160): `in_use` goes to 75, `waiting` stays 2.
- t=80: job 8 (20 nodes, 90 s) fits in the free 25 but 80 + 90 = 170 is not < 160. Waits.

`releases` only covers `(now, shadow_time]` and is empty when nothing waits. To know when a
platform's *next* completion happens when its queue is empty, there is no query today; the
a next-event query would be a small extension.

In [6]:
sim.advance_to(1e9)   # drain everything
s = sim.get_statistics()
print(s)
{k: getattr(s, k) for k in ["jobs_submitted", "jobs_completed", "jobs_running", "jobs_waiting",
                             "current_time", "avg_wait_time", "avg_turnaround_time", "makespan", "utilization"]}

Statistics(jobs=10/10, utilization=66.518987%)


{'jobs_submitted': 10,
 'jobs_completed': 10,
 'jobs_running': 0,
 'jobs_waiting': 0,
 'current_time': 1000000000.0,
 'avg_wait_time': 151.0,
 'avg_turnaround_time': 324.0,
 'makespan': 790.0,
 'utilization': 0.6651898734177215}

`avg_wait_time` 151 and `makespan` 790 match the CLI run in notebook 01. Now the trap: take
the statistics **mid-run**.

In [7]:
params = make_params(100)
sim = dr_evt.Simulation(params)
for i, row in trace.iterrows():
    sim.append_job(float(row.job_submit_time), int(row.num_nodes), QUEUE, float(row.time_limit))
sim.advance_to(100.0)
s = sim.get_statistics()
print(f"t=100: jobs_completed={s.jobs_completed}, jobs_running={s.jobs_running}, jobs_waiting={s.jobs_waiting}")
print(f"       avg_wait_time={s.avg_wait_time:.1f}, makespan={s.makespan:.0f}, utilization={s.utilization:.3f}")

t=100: jobs_completed=0, jobs_running=4, jobs_waiting=6
       avg_wait_time=0.0, makespan=320, utilization=0.431


Nothing has finished at t=100 (`jobs_completed` is 0, and it is honest), yet `makespan`
is 320 and `utilization` is 0.43: the averages, makespan and utilization are computed over
every job that has **started**, using its projected end, and `utilization` divides by
`total_nodes * makespan` with the origin at 0. Rule: never derive service metrics from
`Statistics`; use per-job records. Which brings us to what the in-process API cannot do.

## What you cannot get in process

1. **Per-job start and end times.** `Simulation::get_trace()` exists in C++ but is not
   bound, and there is no `get_job(job_idx)`. Wait time and bounded slowdown have no source.
2. **The simulated-trace CSV.** `write_simulated_trace()` writes to `SimParams.outfile`,
   which cannot be set from Python, so it silently writes nothing.
3. **`run_until_exclusive`** iterates an unused member and is a no-op. Only use
   `advance_to`.
4. **Actual runtime different from the limit.** Appended jobs run exactly `time_limit`
   whatever `run_time_mode` says. "Perfect" runtime knowledge is native; "the user
   over-estimated" needs an actual-runtime field on append, a small extension.
5. **Silent drops and truncation.** A job larger than `total_nodes` is dropped with a
   line on stderr and no exception; `limit_time` is truncated to whole seconds; a wrong
   queue string raises.

Every one of these is demonstrated below.

In [8]:
print("job-related methods:", [n for n in dir(sim) if "job" in n or "trace" in n])
print("has get_job:", hasattr(sim, "get_job"))

job-related methods: ['append_job', 'append_jobs', 'get_active_job_count', 'get_trace_size', 'initialize_trace', 'write_simulated_trace']
has get_job: False


In [9]:
cwd_before = os.getcwd()
scratch = Path(tempfile.mkdtemp(dir=OUT, prefix="scratch_"))
os.chdir(scratch)
sim.write_simulated_trace()
print("files written by write_simulated_trace():", sorted(p.name for p in scratch.iterdir()))
os.chdir(cwd_before)

files written by write_simulated_trace(): []


In [10]:
p2 = make_params(100)
s2 = dr_evt.Simulation(p2)
s2.append_job(0.0, 10, QUEUE, 100.0)
s2.append_job(50.0, 10, QUEUE, 100.0)
s2.advance_to(0.0)
s2.run_until_exclusive(50.0)
print("after run_until_exclusive(50): t =", s2.get_current_time(), "(unchanged: no-op)")
s2.advance_to(50.0)
print("after advance_to(50):          t =", s2.get_current_time(), "| in use =", s2.get_nodes_in_use())

after run_until_exclusive(50): t = 0.0 (unchanged: no-op)
after advance_to(50):          t = 50.0 | in use = 20


In [11]:
p3 = make_params(100)
s3 = dr_evt.Simulation(p3)
s3.append_job(0.0, 10, QUEUE, 100.9)
s3.advance_to(1e6)
print("turnaround of a job appended with limit 100.9:", s3.get_statistics().avg_turnaround_time, "(truncated to 100)")

turnaround of a job appended with limit 100.9: 100.0 (truncated to 100)


In [12]:
p4 = make_params(100)
s4 = dr_evt.Simulation(p4)
idx = s4.append_job(0.0, 200, QUEUE, 100.0)      # 200 nodes on a 100-node cluster
s4.advance_to(10.0)
print(f"oversize job: returned job_idx={idx}, waiting={s4.get_active_job_count()}, in use={s4.get_nodes_in_use()}, "
      f"jobs_submitted={s4.get_statistics().jobs_submitted}  <- dropped silently (a line went to stderr)")
for bad in [(10.0, 1, "gpu-queue", 10.0), (5.0, 1, QUEUE, 10.0)]:
    try:
        s4.append_job(*bad)
    except Exception as e:
        print(f"append_job{bad} -> {type(e).__name__}: {e}")

oversize job: returned job_idx=0, waiting=0, in use=0, jobs_submitted=0  <- dropped silently (a line went to stderr)
append_job(10.0, 1, 'gpu-queue', 10.0) -> ValueError: Invalid queue id: gpu-queue
append_job(5.0, 1, '1', 10.0) -> RuntimeError: Cannot append job with submit_time < current_time. submit_time=5.000000 but current_time=10.000000


Job 0 rejected: requests 200 nodes, exceeds total_nodes (100); this job can never be scheduled.


## Several platforms in one process

Nothing stops you holding N simulations. Two cautions from the source: keep them
single-threaded (`Job_Record::num_inputs` is a process-wide static and the trace parser
writes the `TZ` environment variable), and keep each `SimParams` alive as long as its
`Simulation`.

dr_evt has no speed model, so heterogeneity is ours: the same job gets a different
`limit_time` on each platform, scaled by that platform's relative runtime. The slices below
are the benchmark's federation shares of the real machines (dane 308, lassen 159, tioga 16,
tuolumne 230) with rough relative runtimes from our platform model (tioga ~6x, tuolumne ~8x
faster than lassen; dane ~2x slower).

In [13]:
class Platform:
    # One dr_evt Simulation standing in for one federated platform slice.
    def __init__(self, name, total_nodes, rel_runtime):
        self.name, self.rel_runtime = name, rel_runtime
        self.params = make_params(total_nodes)      # kept alive on purpose
        self.sim = dr_evt.Simulation(self.params)

    def submit(self, t, nodes, runtime_on_lassen_s):
        return self.sim.append_job(t, nodes, QUEUE, runtime_on_lassen_s * self.rel_runtime)

fleet = [Platform("dane", 308, 2.0), Platform("lassen", 159, 1.0),
         Platform("tioga", 16, 1 / 6), Platform("tuolumne", 230, 1 / 8)]
for p in fleet:
    p.submit(0.0, 8, 3600.0)       # the same 8-node job that takes one hour on lassen
    p.sim.advance_to(1e6)
pd.DataFrame([{"platform": p.name, "slice nodes": p.params.total_nodes, "relative runtime": p.rel_runtime,
               "job finished at (s)": p.sim.get_statistics().makespan} for p in fleet])

,platform,slice nodes,relative runtime,job finished at (s)
0,dane,308,2.000000,7200.0
1,lassen,159,1.000000,3600.0
2,tioga,16,0.166667,600.0
3,tuolumne,230,0.125000,450.0


## Determinism

With `run_time_mode=limit` dr_evt uses no randomness, EASY is deterministic given the
submission order, and same-time events tie-break by `job_idx`. The same stream must give
the same numbers every time. This is what lets us pin event-log hashes again in the new
stack.

In [14]:
def run_stream(seed_order):
    p = make_params(100)
    s = dr_evt.Simulation(p)
    for _, row in trace.iloc[seed_order].iterrows():
        s.append_job(float(row.job_submit_time), int(row.num_nodes), QUEUE, float(row.time_limit))
    s.advance_to(1e9)
    st = s.get_statistics()
    return (st.avg_wait_time, st.avg_turnaround_time, st.makespan, st.utilization)

a, b = run_stream(list(range(10))), run_stream(list(range(10)))
print("identical across runs:", a == b, a)

identical across runs: True (151.0, 324.0, 790.0, 0.6651898734177215)


## Exercises

1. You append a job at the current time and immediately read `get_nodes_in_use()`. What
   do you see, and what call is missing?
2. At t=80 in the step-through, 25 nodes were free and job 8 needed 20. Why did it wait?
3. Why is `run_time_mode=limit` the only mode a streaming client can use today, and which
   extension would change that?
4. Name three things a client must validate itself before calling `append_job`.
5. Our auction needs each placed job's wait time. List the two ways to get it today and
   the one-line reason each is unsatisfying.